
# TEST NOTEBOOK (CLEAN & COMPLETE)

This notebook contains:
- Data fetch
- Feature engineering (ML-only)
- Model inference (no leakage)
- Strategy context (RSI/EMA/VWAP)
- Backtesting engine
- Metrics
- Equity curve
- Paper trading skeleton (safe)

This is the **single source of truth**.


In [1]:

import yfinance as yf
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from datetime import datetime


In [2]:

# ==========================
# CONFIG
# ==========================
SYMBOL = "^NSEBANK"
INTERVAL = "1m"
LOOKBACK = "7d"
CAPITAL = 100000
PROB_TH = 0.55


In [3]:

# ==========================
# DATA FETCH
# ==========================
df = yf.download(SYMBOL, interval=INTERVAL, period=LOOKBACK)
df.dropna(inplace=True)
df


C:\Users\Nihar\AppData\Local\Temp\ipykernel_13172\1684435438.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(SYMBOL, interval=INTERVAL, period=LOOKBACK)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK
Datetime,,,,,
2025-12-22 03:45:00+00:00,59215.449219,59258.250000,59158.898438,59224.750000,0
2025-12-22 03:46:00+00:00,59239.699219,59239.699219,59206.500000,59217.601562,0
2025-12-22 03:47:00+00:00,59242.550781,59261.699219,59225.949219,59231.300781,0
2025-12-22 03:48:00+00:00,59248.648438,59268.750000,59239.300781,59239.300781,0
2025-12-22 03:49:00+00:00,59254.601562,59263.699219,59244.449219,59247.949219,0
...,...,...,...,...,...
2025-12-30 09:20:00+00:00,59151.101562,59159.398438,59140.000000,59155.550781,0
2025-12-30 09:21:00+00:00,59160.050781,59169.750000,59140.398438,59151.250000,0


In [4]:

# ==========================
# ML FEATURE ENGINEERING
# (ONLY what model was trained on)
# ==========================
df["ret"] = df["Close"].pct_change()
df["vol"] = df["Volume"].pct_change()
df.dropna(inplace=True)

X = df[["ret", "vol"]]


In [6]:

# ==========================
# LOAD MODEL + PREDICTION
# ==========================
model = joblib.load("model.pkl")

df["pred_prob"] = model.predict_proba(X)[:, 1]

# CRITICAL: shift prediction
df["pred_prob"] = df["pred_prob"].shift(1)
df.dropna(inplace=True)


c:\Users\Nihar\Documents\GitHub\.venv\Lib\site-packages\xgboost\core.py:160: UserWarning: [15:01:28] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0750514818a16474a-1\xgboost\xgboost-ci-windows\src\common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)


AttributeError: 'dict' object has no attribute 'predict_proba'

In [ ]:

# ==========================
# STRATEGY CONTEXT
# (NOT ML FEATURES)
# ==========================
def add_strategy_context(df):
    df = df.copy()

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["rsi"] = 100 - (100 / (1 + rs))

    df["ema9"] = df["Close"].ewm(span=9).mean()
    df["ema21"] = df["Close"].ewm(span=21).mean()

    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    df["vwap"] = (tp * df["Volume"]).cumsum() / df["Volume"].cumsum()

    return df

df = add_strategy_context(df)
df.dropna(inplace=True)


In [ ]:

# ==========================
# STRATEGY FILTER
# ==========================
def strategy_filter(row):
    if row["pred_prob"] < PROB_TH:
        return 0

    if row["ema9"] > row["ema21"] and row["rsi"] < 70 and row["Close"] > row["vwap"]:
        return 1

    return 0

df["signal"] = df.apply(strategy_filter, axis=1)


In [ ]:

# ==========================
# BACKTEST ENGINE
# ==========================
capital = CAPITAL
position = 0
entry_price = 0
entry_time = None
equity = capital
equity_curve = []
trades = []

for i in range(1, len(df)):
    row = df.iloc[i]

    equity_curve.append(equity)

    if position == 0 and row["signal"] == 1:
        position = 1
        entry_price = row["Open"]
        entry_time = row.name

    elif position == 1:
        exit_cond = (
            row["ema9"] < row["ema21"] or
            row["rsi"] > 75
        )

        if exit_cond:
            exit_price = row["Open"]
            ret = (exit_price - entry_price) / entry_price
            equity *= (1 + ret)

            trades.append({
                "entry_time": entry_time,
                "exit_time": row.name,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "return": ret,
                "equity": equity
            })

            position = 0

equity_curve = pd.Series(equity_curve, index=df.index[:len(equity_curve)])


In [ ]:

# ==========================
# METRICS
# ==========================
trades_df = pd.DataFrame(trades)

total_trades = len(trades_df)
win_rate = (trades_df["return"] > 0).mean()
avg_return = trades_df["return"].mean()
cum_return = equity / CAPITAL - 1

print("Trades:", total_trades)
print("Win rate:", round(win_rate, 3))
print("Avg return:", round(avg_return, 4))
print("Cumulative return:", round(cum_return, 4))


In [ ]:

# ==========================
# EQUITY CURVE
# ==========================
plt.figure(figsize=(10,4))
equity_curve.plot()
plt.title("Equity Curve")
plt.grid()
plt.show()


In [ ]:

# ==========================
# PAPER TRADING SKELETON
# ==========================
# Same pipeline as backtest
# Run this cell only in live market hours

def run_paper_trade(latest_df):
    latest_df = add_strategy_context(latest_df)
    latest_df["ret"] = latest_df["Close"].pct_change()
    latest_df["vol"] = latest_df["Volume"].pct_change()
    latest_df.dropna(inplace=True)

    X_live = latest_df[["ret", "vol"]].iloc[-1:]
    prob = model.predict_proba(X_live)[:, 1][0]

    row = latest_df.iloc[-1]

    if prob >= PROB_TH and row["ema9"] > row["ema21"] and row["rsi"] < 70:
        return "BUY"
    return "HOLD"
